# Stockout Trainer — Notebook (inline training code)

Ce notebook contient le code d'entraînement complet (copié depuis app/scripts/train_model.py) afin de pouvoir l'exécuter directement sur Kaggle sans importer le module.
- Run All exécutera un entraînement réduit (200 produits × 120 jours) pour test.
- Les artefacts sont écrits dans app/models_artifacts/.


In [ ]:
# Installer dépendances
!pip install --quiet xgboost scikit-learn pandas numpy joblib nbformat

In [ ]:
# Préparer sys.path pour permettre les imports locaux (au cas où)
import os, sys
cw = os.getcwd()
candidates = [cw, os.path.join(cw, 'stockout-modern'), os.path.join(cw, 'stockout-modern','backend'), os.path.join(cw,'backend')]
for p in candidates:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)
print('sys.path prepared; first entries:', sys.path[:5])


In [ ]:
# Code d'entraînement complet (adapté pour notebook)
import os, sys, json, logging
import numpy as np
logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(message)s')
logger = logging.getLogger('__notebook_train__')
# Emplacement des artefacts : app/models_artifacts relative au repo layout attendu
# Cherche candidate backend/app si présent dans cwd tree
repo_backend = None
for cand in [os.path.join(os.getcwd(),'stockout-modern','backend'), os.path.join(os.getcwd(),'backend'), os.getcwd()]:
    if os.path.isdir(os.path.join(cand,'app')):
        repo_backend = cand
        break
if repo_backend is None:
    repo_backend = os.getcwd()
MODEL_DIR = os.path.join(repo_backend, 'app', 'models_artifacts')
os.makedirs(MODEL_DIR, exist_ok=True)
FEATURES = [
    'lag_1', 'lag_7', 'lag_14', 'lag_30',
    'rolling_mean_7', 'rolling_mean_14',
    'rolling_std_7', 'rolling_std_14',
    'current_stock', 'safety_stock', 'stock_to_demand_ratio',
    'lead_time', 'day_of_week', 'month', 'horizon',
]
def train(n_products: int = 300, n_days: int = 365, dataset: str = None, dataset_format: str = None) -> dict:
    try:
        import pandas as pd
        import xgboost as xgb
        import joblib
        from sklearn.preprocessing import StandardScaler
        from sklearn.model_selection import train_test_split
        from sklearn.metrics import roc_auc_score, brier_score_loss
    except ImportError as e:
        logger.error(f'Dépendance manquante : {e}')
        raise

    # If dataset provided, load and map to features
    if dataset is not None:
        logger.info(f'Chargement dataset fourni: {dataset} (format={dataset_format})')
        if dataset_format == 'm5' or (dataset_format is None and os.path.isdir(dataset)):
            # expect a folder with sales_train_validation.csv and calendar.csv
            try:
                from app.scripts.mapper_m5 import m5_to_raw
            except Exception as e:
                logger.error('mapper_m5 introuvable — assurez-vous que stockout-modern/backend est présent dans le notebook workspace')
                raise
            raw = m5_to_raw(dataset, max_products=n_products, max_days=n_days)
            try:
                from app.scripts.generate_synthetic_data import build_features
            except Exception:
                logger.error('generate_synthetic_data introuvable — assurez-vous que le repo est complet')
                raise
            df = build_features(raw, horizons=[7,14,30])
        else:
            if dataset.endswith('.parquet'):
                raw_df = pd.read_parquet(dataset)
            else:
                raw_df = pd.read_csv(dataset, parse_dates=['date'])
            from app.scripts.generate_synthetic_data import build_features as build_features_generic
            df = build_features_generic(raw_df, horizons=[7,14,30])
    else:
        # use synthetic generator as fallback
        try:
            from app.scripts.generate_synthetic_data import generate_dataset, build_features
        except Exception as e:
            logger.error('generate_synthetic_data introuvable — cannot continue')
            raise
        logger.info(f'Génération données synthétiques — {n_products} produits × {n_days} jours...')
        raw = generate_dataset(n_products=n_products, n_days=n_days)
        df = build_features(raw, horizons=[7,14,30])

    pos_rate = df['stockout'].mean()
    logger.info(f'Dataset : {len(df):,} exemples | {pos_rate:.1%} ruptures')

    X = df[FEATURES].values.astype(np.float32)
    y = df['stockout'].values.astype(np.int32)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    neg = (y_train == 0).sum()
    pos = (y_train == 1).sum()
    scale_pos = neg / max(pos, 1)
    logger.info(f'scale_pos_weight = {scale_pos:.2f}  (neg={neg}, pos={pos})')

    logger.info('Entraînement XGBoost...')
    model = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        gamma=1.0,
        scale_pos_weight=scale_pos,
        random_state=42,
        eval_metric='auc',
        early_stopping_rounds=30,
        verbosity=0,
    )
    model.fit(
        X_train_s, y_train,
        eval_set=[(X_test_s, y_test)],
        verbose=False,
    )

    proba = model.predict_proba(X_test_s)[:,1]
    from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, roc_auc_score, brier_score_loss
    auc = roc_auc_score(y_test, proba)
    brier = brier_score_loss(y_test, proba)
    thresh = 0.5
    preds = (proba >= thresh).astype(int)
    precision = precision_score(y_test, preds, zero_division=0)
    recall = recall_score(y_test, preds, zero_division=0)
    f1 = f1_score(y_test, preds, zero_division=0)
    pr_auc = average_precision_score(y_test, proba)

    logger.info(f'AUC    = {auc:.4f}  (>0.75 = bon, >0.85 = excellent)')
    logger.info(f'PR-AUC = {pr_auc:.4f}')
    logger.info(f'Brier  = {brier:.4f}  (<0.15 = bon, <0.10 = excellent)')
    logger.info(f'Precision/Recall/F1 @0.5 = {precision:.3f} / {recall:.3f} / {f1:.3f}')

    importances = dict(zip(FEATURES, model.feature_importances_))
    top5 = sorted(importances.items(), key=lambda x: -x[1])[:5]
    logger.info('Top 5 features : ' + ' | '.join(f'{k}={v:.3f}' for k,v in top5))

    joblib.dump(model, os.path.join(MODEL_DIR, 'xgb_model.pkl'))
    joblib.dump(scaler, os.path.join(MODEL_DIR, 'scaler.pkl'))
    joblib.dump(FEATURES, os.path.join(MODEL_DIR, 'feature_names.pkl'))

    metrics = {
        'auc': round(auc,4),
        'pr_auc': round(pr_auc,4),
        'brier': round(brier,4),
        'precision': round(precision,4),
        'recall': round(recall,4),
        'f1': round(f1,4),
        'n_train': len(X_train),
        'n_test': len(X_test),
        'pos_rate': round(float(pos_rate),4),
        'best_iteration': getattr(model,'best_iteration',None),
    }
    with open(os.path.join(MODEL_DIR,'metrics.json'),'w') as f:
        json.dump(metrics,f,indent=2)
    metrics_fr = {
        'AUC': metrics['auc'],
        'PR_AUC': metrics['pr_auc'],
        'Score de Brier': metrics['brier'],
        'Précision': metrics['precision'],
        'Rappel': metrics['recall'],
        'F1': metrics['f1'],
        'Exemples entraînement': metrics['n_train'],
        'Exemples test': metrics['n_test'],
        'Taux de positifs': metrics['pos_rate'],
    }
    with open(os.path.join(MODEL_DIR,'metrics_fr.json'),'w',encoding='utf-8') as f:
        json.dump(metrics_fr,f,ensure_ascii=False,indent=2)

    try:
        import datetime as _dt
        version_str = _dt.datetime.utcnow().isoformat() + 'Z'
        with open(os.path.join(MODEL_DIR,'version.txt'),'w') as vf:
            vf.write(version_str)
    except Exception:
        pass

    logger.info(f'Modèles sauvegardés dans {os.path.abspath(MODEL_DIR)}')
    return metrics


In [ ]:
# Lancer un entraînement réduit pour test
print('Démarrage entraînement réduit (200 produits, 120 jours)')
metrics = train(n_products=200, n_days=120, dataset=None, dataset_format=None)
print('
Metrics:', metrics)


In [ ]:
# Afficher les artefacts produits
import os, json
art_dir = os.path.join(repo_backend, 'app', 'models_artifacts')
print('Artifacts in', art_dir)
if os.path.exists(art_dir):
    print('
'.join(sorted(os.listdir(art_dir))))
    m_path = os.path.join(art_dir,'metrics.json')
    if os.path.exists(m_path):
        print('--- metrics.json ---')
        print(json.dumps(json.load(open(m_path)), indent=2))
else:
    print('Aucun artifact trouvé')
